

| # | Município | Câmara (URL base candidata) | População aprox. |
|---|-----------|----------------------------|------------------|
| 1 | João Pessoa     | https://www.cmjp.pb.gov.br/          | ~833 mil |
| 2 | Campina Grande  | https://www.campinagrande.pb.leg.br/ | ~411 mil |
| 3 | Santa Rita      | https://www.santarita.pb.leg.br/     | ~136 mil |
| 4 | Patos           | https://www.patos.pb.leg.br/         | ~107 mil |
| 5 | Bayeux          | https://www.bayeux.pb.leg.br/        | ~94 mil  |




Instala as dependências do notebook.  


In [1]:

%pip install -q requests beautifulsoup4 lxml pandas tenacity tqdm

Note: you may need to restart the kernel to use updated packages.


Imports e configuração global do logger. Logging em vez de `print` para facilitar debugar quando algum portal mudar de layout (um dos riscos identificados na pesquisa).

In [2]:
from __future__ import annotations

import json
import logging
import re
import sqlite3
import time
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Iterable
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser
import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from tqdm.auto import tqdm
from urllib3.util.retry import Retry

/Users/chaves/programacao/Trainees_2026.1-Pauta/scrapers/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
logging.basicConfig(level=logging.INFO,format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",datefmt="%H:%M:%S")
log = logging.getLogger("pauta.scraper")

OUTPUT_DIR = Path("data")
OUTPUT_DIR.mkdir(exist_ok=True)
log.info("Outputs serão salvos em %s/", OUTPUT_DIR.resolve())

16:36:37 [INFO] pauta.scraper — Outputs serão salvos em /Users/chaves/programacao/Trainees_2026.1-Pauta/scrapers/data/


Padrões nas câmaras da Paraíba:
- **João Pessoa** usa sistema próprio (`cmjp.pb.gov.br`).
- As demais costumam estar sob o domínio padronizado **Interlegis** (`<municipio>.pb.leg.br`), normalmente rodando SAPL ou Portal Modelo — o que ajuda a generalizar os extratores.


In [4]:
@dataclass
class CamaraConfig:
    slug: str
    municipio: str
    uf: str = "PB"
    base_url: str = ""
    url_vereadores: str | None = None   # URL direta (descoberta manual via inspeção)
    url_projetos:   str | None = None
    url_noticias:   str | None = None
    sistema: str = ""
    sapl_base: str | None = None         # se a câmara usa SAPL, URL raiz da instância
    url_json_projetos: str | None = None # alternativa: página que embute JSON inline (caso Patos)                    # "Interlegis", "WordPress", "SAPL", "próprio"
    observacoes: str = ""


CIDADES: list[CamaraConfig] = [
    CamaraConfig(
        slug="joao_pessoa",
        municipio="João Pessoa",
        base_url="https://joaopessoa.pb.leg.br",
        url_vereadores="https://joaopessoa.pb.leg.br/vereadores",
        url_projetos="https://sapl.joaopessoa.pb.leg.br/materia/pesquisar-materia",
        url_noticias="https://joaopessoa.pb.leg.br/noticias/",
        sistema="Interlegis (SAPL) + tema próprio",
        sapl_base="https://sapl.joaopessoa.pb.leg.br",
    ),
    CamaraConfig(
        slug="campina_grande",
        municipio="Campina Grande",
        base_url="https://www.camaracg.pb.gov.br",
        url_vereadores="https://www.camaracg.pb.gov.br/vereadores/",
        url_projetos="https://www.camaracg.pb.gov.br/category/projetos-de-lei/",
        url_noticias="https://www.camaracg.pb.gov.br/noticias/",
        sistema="WordPress (tagDiv)",
        sapl_base="https://sapl.campinagrande.pb.leg.br",  # SAPL legado ainda ativo
        observacoes="Domínio antigo .leg.br redireciona pra cá. "
                    "⚠ Não publica partido dos vereadores em nenhum lugar do site — "
                    "para preencher, seria necessária fonte externa (TSE).",
    ),
    CamaraConfig(
        slug="santa_rita",
        municipio="Santa Rita",
        base_url="https://www.santarita.pb.leg.br",
        url_vereadores="https://www.santarita.pb.leg.br/site/vereadores/",
        url_projetos="https://sapl.santarita.pb.leg.br/materia/pesquisar-materia?salvar=Pesquisar",
        url_noticias="https://www.santarita.pb.leg.br/site/category/noticias/",
        sistema="WordPress (Elementor) + SAPL externo",
        sapl_base="https://sapl.santarita.pb.leg.br",
        observacoes="robots.txt do site institucional bloqueia bots (depende da flag RESPECT_ROBOTS).",
    ),
    CamaraConfig(
        slug="patos",
        municipio="Patos",
        base_url="http://camarapatos.pb.gov.br",
        url_vereadores="https://camarapatos.pb.gov.br/a-camara/vereadores",
        url_projetos="https://camarapatos.pb.gov.br/a-camara/Atividades_Legislativas_dos_Parlamentares",
        url_noticias="https://camarapatos.pb.gov.br/informativos/noticias",
        sistema="próprio (CamaraNova)",
        url_json_projetos="https://camarapatos.pb.gov.br/consultas/projetodeleiprotocolados",
        observacoes="Vereadores via JS (não scrapeable); projetos via JSON inline na página de protocolo.",
    ),
    CamaraConfig(
        slug="bayeux",
        municipio="Bayeux",
        base_url="https://bayeux.pb.leg.br",
        url_vereadores="https://bayeux.pb.leg.br/parlamentares",
        url_projetos="https://sapl.bayeux.pb.leg.br/materia/pesquisar-materia?salvar=Pesquisar",
        url_noticias="https://bayeux.pb.leg.br/noticias/",
        sistema="SAPL (Interlegis)",
        sapl_base="https://sapl.bayeux.pb.leg.br",
    ),
]

pd.DataFrame([asdict(c) for c in CIDADES])[["slug", "municipio", "sistema", "base_url"]]


,slug,municipio,sistema,base_url
0,joao_pessoa,João Pessoa,Interlegis (SAPL) + tema próprio,https://joaopessoa.pb.leg.br
1,campina_grande,Campina Grande,WordPress (tagDiv),https://www.camaracg.pb.gov.br
2,santa_rita,Santa Rita,WordPress (Elementor) + SAPL externo,https://www.santarita.pb.leg.br
3,patos,Patos,próprio (CamaraNova),http://camarapatos.pb.gov.br
4,bayeux,Bayeux,SAPL (Interlegis),https://bayeux.pb.leg.br



CLIENTE HTTP:


In [5]:
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


USER_AGENT = (
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/127.0.0.0 Safari/537.36"
)
REQUEST_TIMEOUT = 20           
POLITE_DELAY_SECONDS = 1.2     
VERIFY_SSL = False           
RESPECT_ROBOTS = False         


def build_session() -> requests.Session:
    """Sessão com pooling + retry para 5xx/429 no nível do urllib3."""
    sess = requests.Session()
    sess.headers.update({
        "User-Agent": USER_AGENT,
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Language": "pt-BR,pt;q=0.9,en;q=0.5",
        "Accept-Encoding": "gzip, deflate",  
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1",
    })
    retry = Retry(
        total=3, backoff_factor=1.5,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=("GET", "HEAD"),
        raise_on_status=False,
    )
    sess.mount("https://", HTTPAdapter(max_retries=retry, pool_connections=10, pool_maxsize=10))
    sess.mount("http://",  HTTPAdapter(max_retries=retry))
    sess.verify = VERIFY_SSL
    return sess


SESSION = build_session()
_ROBOTS_CACHE: dict[str, RobotFileParser] = {}


def _robots_for(url: str) -> RobotFileParser:
    parsed = urlparse(url)
    host = f"{parsed.scheme}://{parsed.netloc}"
    if host in _ROBOTS_CACHE:
        return _ROBOTS_CACHE[host]
    rp = RobotFileParser()
    rp.set_url(urljoin(host, "/robots.txt"))
    try:
        rp.read()
    except Exception as exc:
        log.warning("robots.txt inacessível em %s (%s) — assumindo permitido", host, exc)
    _ROBOTS_CACHE[host] = rp
    return rp


def can_fetch(url: str) -> bool:
    if not RESPECT_ROBOTS:
        return True
    try:
        return _robots_for(url).can_fetch(USER_AGENT, url)
    except Exception:
        return True  


class FetchError(RuntimeError):
    pass


@retry(
    reraise=True,
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=1, max=8),
    retry=retry_if_exception_type((requests.ConnectionError, requests.Timeout)),
)
def fetch(url: str, *, allow_404: bool = False) -> str | None:
    """GET seguindo redirects, com User-Agent realista, robots.txt e delay polite."""
    if not can_fetch(url):
        log.warning("robots.txt PROIBE: %s — pulando", url)
        return None
    time.sleep(POLITE_DELAY_SECONDS)
    try:
        resp = SESSION.get(url, timeout=REQUEST_TIMEOUT, allow_redirects=True)
    except requests.exceptions.SSLError as exc:
        # fallback explícito (já vem com verify=False, mas por segurança)
        log.warning("SSL falhou em %s (%s) — tentando sem verificação", url, exc)
        resp = SESSION.get(url, timeout=REQUEST_TIMEOUT, allow_redirects=True, verify=False)

    if resp.status_code == 404 and allow_404:
        return None
    if resp.status_code >= 400:
        raise FetchError(f"HTTP {resp.status_code} em {url} (final: {resp.url})")
    if not resp.encoding or resp.encoding.lower() == "iso-8859-1":
        resp.encoding = resp.apparent_encoding
    return resp.text


def soup_of(html: str) -> BeautifulSoup:
    return BeautifulSoup(html, "lxml")


log.info("Cliente HTTP pronto. UA=Chrome127 · verify_ssl=%s · respect_robots=%s", VERIFY_SSL, RESPECT_ROBOTS)


16:36:37 [INFO] pauta.scraper — Cliente HTTP pronto. UA=Chrome127 · verify_ssl=False · respect_robots=False


TESTANDO ENDPOINTS PARA VERIFICAR VALDIADE:



In [6]:

def validar_endpoints(cidade: CamaraConfig) -> dict:
    resultado = {
        "slug": cidade.slug,
        "municipio": cidade.municipio,
        "sistema": cidade.sistema,
    }
    for nome, url in [
        ("vereadores", cidade.url_vereadores),
        ("projetos",   cidade.url_projetos),
        ("noticias",   cidade.url_noticias),
    ]:
        if not url:
            resultado[nome] = None
            continue
        try:
            html = fetch(url, allow_404=True)
            ok = html is not None and len(html) > 1000  # heurística mínima de "página real"
            resultado[nome] = "OK" if ok else "VAZIO"
        except FetchError as exc:
            resultado[nome] = f"ERR {exc}"[:30]
            log.warning("%s/%s falhou: %s", cidade.slug, nome, exc)
    return resultado


descobertas = [validar_endpoints(c) for c in tqdm(CIDADES, desc="Validando endpoints")]
df_disc = pd.DataFrame(descobertas)
df_disc


Validando endpoints:   0%|          | 0/5 [00:00<?, ?it/s]

Validando endpoints:  20%|██        | 1/5 [00:10<00:40, 10.09s/it]

Validando endpoints:  40%|████      | 2/5 [00:25<00:40, 13.43s/it]

Validando endpoints:  60%|██████    | 3/5 [00:37<00:24, 12.49s/it]

Validando endpoints:  80%|████████  | 4/5 [00:41<00:09,  9.14s/it]

Validando endpoints: 100%|██████████| 5/5 [00:50<00:00,  9.09s/it]

Validando endpoints: 100%|██████████| 5/5 [00:50<00:00, 10.05s/it]

,slug,municipio,sistema,vereadores,projetos,noticias
0,joao_pessoa,João Pessoa,Interlegis (SAPL) + tema próprio,OK,OK,OK
1,campina_grande,Campina Grande,WordPress (tagDiv),OK,OK,OK
2,santa_rita,Santa Rita,WordPress (Elementor) + SAPL externo,OK,OK,OK
3,patos,Patos,próprio (CamaraNova),OK,OK,OK
4,bayeux,Bayeux,SAPL (Interlegis),OK,OK,OK


Cada portal lista vereadores de um jeito (cards, tabela, grid). Em vez de escrever 5 extratores diferentes do zero, usamos uma **heurística genérica**: identificar repetições de blocos com estrutura similar (cards) e extrair, dentro de cada bloco, o nome do vereador + qualquer campo evidente (partido, contato).

Heurística:
1. Encontrar elementos repetidos (`div`/`li`/`article`) cuja classe contenha palavras como `card`, `vereador`, `parlamentar`, `member`.
2. Em cada bloco, extrair: heading (h1-h4) como nome, sigla de partido por regex (`PSDB`, `PT`, `MDB`, ...), e-mail/telefone se houver.
3. Fallback: se nada repetido for encontrado, tentar tabela (`<table>`).



In [7]:


RX_PARTIDO = re.compile(
    r"\b(PT|PSDB|MDB|PSB|PSD|PP|PL|PDT|REPUBLICANOS|UNIÃO|UNIAO|PODEMOS|"
    r"AVANTE|PCdoB|PV|REDE|NOVO|CIDADANIA|SOLIDARIEDADE|PSOL|PRTB|PMN|PROS|AGIR|MOBILIZA)\b",
    re.IGNORECASE,
)
RX_EMAIL = re.compile(r"[\w\.\-+]+@[\w\-]+\.[\w\.\-]+")
RX_FONE  = re.compile(r"\(?\d{2}\)?\s?\d{4,5}[-\s]?\d{4}")
RX_REDES = re.compile(r"https?://[^/\s\"']*(instagram|facebook|twitter|x\.com|tiktok|youtube)\.com[^\s\"']*", re.I)


def _txt(tag):
    return re.sub(r"\s+", " ", tag.get_text(" ", strip=True)) if tag else ""


def _v_joao_pessoa(soup, base_url):
    """JP: cada vereador é o container PAI de div.vereador-dados.
    Inclui foto, telefone, email (renderizado de forma esquisita com <i>fa-at</i>).
    URL do 'perfil' é uma tag de notícias /tag/<slug>/."""
    out = []
    for dados in soup.select("div.vereador-dados"):
        card = dados.find_parent("div", class_="col-12")
        if not card: card = dados.parent
        spans = dados.find_all("span")
        nome    = _txt(spans[0]) if spans else ""
        partido = _txt(spans[1]) if len(spans) > 1 else None
        if not nome: continue
        # contato
        contato = card.select_one("div.vereador-contato")
        texto_c = _txt(contato)
        m_email = RX_EMAIL.search(texto_c.replace(" ", ""))
        email = m_email.group(0) if m_email else None
        if not email and contato:
            partes = re.findall(r"(\S+)\s*", texto_c)
            for p_ in partes:
                if "joaopessoa.pb.leg.br" in p_ and not p_.startswith("@"):
                    email = p_.lstrip("@"); break
        # foto + link de notícias
        img = card.find("img")
        a   = card.find("a", href=True)
        out.append({
            "nome": nome, "partido": partido,
            "telefone": (RX_FONE.search(texto_c) or [None])[0],
            "email":    email,
            "foto":     img["src"] if img else None,
            "url_perfil": a["href"] if a else None,
            "bio": None,
        })
    return out


def _v_campina_grande(soup, base_url):
    """CG: card td_module_2, paginado em /vereadores/page/N/. Segue link de perfil pra bio."""
    out = []
   
    todos_cards = list(soup.select("div.td_module_2"))
    for pg in range(2, 8):
        url_pg = f"{base_url.rstrip('/')}/page/{pg}/"
        try:
            html_pg = fetch(url_pg, allow_404=True)
        except FetchError:
            break
        if not html_pg: break
        cards_pg = soup_of(html_pg).select("div.td_module_2")
        if not cards_pg: break
        todos_cards.extend(cards_pg)
    for card in todos_cards:
        a = card.find("a", title=True)
        if not a: continue
        nome = a["title"].strip()
        if not nome: continue
        url_perfil = a.get("href")
        img = card.find("img")
        foto = (img.get("data-img-url") or img.get("src")) if img else None
        bio, partido, email, fone = None, None, None, None
        try:
            html_p = fetch(url_perfil, allow_404=True)
            if html_p:
                sp = soup_of(html_p)
                main = sp.find("main") or sp.find("article") or sp
                bio  = _txt(main)[:2000]
                m_p = RX_PARTIDO.search(bio)
                if m_p: partido = m_p.group(1).upper()
                # contato (CG só tem genérico da câmara — ignorar)
        except Exception as exc:
            log.debug("CG perfil %s falhou: %s", url_perfil, exc)
        out.append({
            "nome": nome, "partido": partido,
            "telefone": fone, "email": email,
            "foto": foto, "url_perfil": url_perfil, "bio": bio,
        })
    return out


def _v_santa_rita(soup, base_url):
    """SR: cada vereador é um div com classes tx_partidos-<slug> e tx_funcoes-<slug>.
    Link de perfil tem padrão /site/pt_vereadores/<slug>/."""
    out = []
    seen = set()
    for div in soup.find_all("div", class_=re.compile(r"pt_vereadores")):
        classes = " ".join(div.get("class") or [])
        # extrai partido pelas classes
        m_p = re.search(r"tx_partidos-([a-z]+)", classes)
        partido = m_p.group(1).upper() if m_p else None
        m_f = re.search(r"tx_funcoes-([a-z\-]+)", classes)
        funcao = m_f.group(1) if m_f else None
        # nome + link
        a = div.find("a", href=re.compile(r"/pt_vereadores/"))
        if not a: continue
        url_perfil = a["href"]
        if url_perfil in seen: continue
        seen.add(url_perfil)
        h3 = div.find("h3")
        nome = _txt(h3) if h3 else (a.get("aria-label") or "").strip()
        if not nome: continue
        img = div.find("img")
        foto = img.get("src") if img else None
        out.append({
            "nome": nome, "partido": partido,
            "telefone": None, "email": None,
            "foto": foto, "url_perfil": url_perfil,
            "funcao": funcao, "bio": None,
        })
    return out


def _v_patos(soup, base_url):
    """Patos não tem vereadores em HTML estático. Workaround: extrair do mesmo JSON
    de projetos que usaremos na Fase 6 — quem aparece como remetente >=2x é candidato
    a vereador. Heurística imperfeita mas é melhor que zero."""
    try:
        html = fetch("https://camarapatos.pb.gov.br/consultas/projetodeleiprotocolados")
    except FetchError as exc:
        log.warning("patos: %s", exc); return []
    m = re.search(r"const\s+projetosDeLei\s*=\s*(\[.*?\]);", html, re.DOTALL)
    if not m: return []
    data = json.loads(m.group(1))
    from collections import Counter
    cnt = Counter((p.get("remetente") or "").strip() for p in data)
    out = []
    for nome, n in cnt.items():
        if n < 2 or not nome: continue   # filtra cidadão isolado
        # filtra entidades genéricas
        if any(k in nome.upper() for k in ["MESA","EXECUTIVO","PREFEITO","COMISSÃO","COMISSAO"]):
            continue
        out.append({
            "nome": nome.title(), "partido": None,
            "telefone": None, "email": None,
            "foto": None, "url_perfil": None, "bio": None,
        })
    log.info("patos: %d candidatos a vereador via remetentes (≥2 projetos)", len(out))
    return out


def _v_bayeux(soup, base_url):
    """Bayeux SAPL: cada sapl-card tem botão abrirModal<ID>() — o sapl-modal-content
    correspondente já está renderizado no HTML com biografia completa."""
    out = []
    # constrói índice modal-id → conteúdo
    modais = {}
    for m in soup.select("div.sapl-modal"):
        mid = m.get("id", "").replace("modal", "")
        cont = m.select_one("div.sapl-modal-content")
        if mid.isdigit() and cont:
            modais[mid] = cont
    for card in soup.select("div.sapl-card"):
        h3 = card.find("h3")
        if not h3: continue
        nome = _txt(h3)
        partido_tag = card.select_one("div.sapl-partido")
        info_tag    = card.select_one("div.sapl-info")
        partido = _txt(partido_tag) or None
        email_match = RX_EMAIL.search(_txt(info_tag).replace(" ", ""))
        email   = email_match.group(0) if email_match else None
        # foto
        img = card.find("img"); foto = img.get("src") if img else None
        # bio do modal correspondente
        btn = card.find("button", onclick=True)
        bio = None
        if btn:
            mid_match = re.search(r"abrirModal(\d+)", btn["onclick"])
            if mid_match and mid_match.group(1) in modais:
                bio = _txt(modais[mid_match.group(1)])[:2000]
        out.append({
            "nome": nome, "partido": partido,
            "telefone": None, "email": email,
            "foto": foto, "url_perfil": None, "bio": bio,
        })
    return out


ADAPTERS_VEREADORES = {
    "joao_pessoa":    _v_joao_pessoa,
    "campina_grande": _v_campina_grande,
    "santa_rita":     _v_santa_rita,
    "patos":          _v_patos,
    "bayeux":         _v_bayeux,
}


def extrair_vereadores(cidade: CamaraConfig) -> list[dict]:
    url = cidade.url_vereadores
    if not url:
        log.warning("%s: sem URL de vereadores", cidade.slug)
        return []
    try:
        html = fetch(url)
    except FetchError as exc:
        log.error("%s: falha (%s)", cidade.slug, exc)
        return []
    soup = soup_of(html)
    adapter = ADAPTERS_VEREADORES.get(cidade.slug)
    if not adapter: return []
    registros = adapter(soup, url)
    for r in registros:
        r["municipio"]  = cidade.municipio
        r["slug"]       = cidade.slug
        r["url_origem"] = url
    log.info("%-15s vereadores: %d (com bio: %d, com email: %d, com telefone: %d)",
             cidade.slug, len(registros),
             sum(1 for r in registros if r.get("bio")),
             sum(1 for r in registros if r.get("email")),
             sum(1 for r in registros if r.get("telefone")))
    return registros


vereadores_raw: list[dict] = []
for c in tqdm(CIDADES, desc="Vereadores"):
    vereadores_raw.extend(extrair_vereadores(c))

df_vereadores_raw = pd.DataFrame(vereadores_raw)
print(f"Total: {len(df_vereadores_raw)} vereadores")
df_vereadores_raw[["municipio","nome","partido","email","telefone"]].head(15)


Vereadores:   0%|          | 0/5 [00:00<?, ?it/s]

16:37:31 [INFO] pauta.scraper — joao_pessoa     vereadores: 28 (com bio: 0, com email: 27, com telefone: 23)


Vereadores:  20%|██        | 1/5 [00:03<00:13,  3.37s/it]

16:39:25 [INFO] pauta.scraper — campina_grande  vereadores: 23 (com bio: 23, com email: 0, com telefone: 0)


Vereadores:  40%|████      | 2/5 [01:57<03:25, 68.35s/it]

...

16:39:36 [INFO] pauta.scraper — bayeux          vereadores: 17 (com bio: 0, com email: 17, com telefone: 0)


Vereadores: 100%|██████████| 5/5 [02:08<00:00, 17.14s/it]

Vereadores: 100%|██████████| 5/5 [02:08<00:00, 25.64s/it]

Total: 97 vereadores


,municipio,nome,partido,email,telefone
0,João Pessoa,Bosquinho,PV,joaopessoa.pb.leg.br,(83) 3218-6323
1,João Pessoa,Carlão,PL,joaopessoa.pb.leg.br,(83) 3218-6326
2,João Pessoa,Chico do Sindicato,Avante,joaopessoa.pb.leg.br,(83) 3218-6318
3,João Pessoa,Damásio Franca,PP,NaN,NaN
4,João Pessoa,Dinho,MDB,joaopessoa.pb.leg.br,(83) 3218-6375
5,João Pessoa,Eliza Virgínia,PP,joaopessoa.pb.leg.br,(83) 3218-6319
6,João Pessoa,Fábio Carneiro,Solidariedade,joaopessoa.pb.leg.br,(83) 3218-6322
7,João Pessoa,Fábio Lopes,PL,joaopessoa.pb.leg.br,(83) 3218-6301
8,João Pessoa,Guga Pet,PP,joaopessoa.pb.leg.br,(83) 3218-6305
9,João Pessoa,Guguinha Moov Jampa,PSD,joaopessoa.pb.leg.br,(83) 3218-6316



D
Para cada matéria extraída, pegamos: `tipo`, `numero`, `ano`, `ementa`, `autor`


| Câmara | Estratégia | Disponibilidade |
|--------|-----------|----------------|
| JP, Santa Rita, Bayeux | adapter SAPL com paginação | ✓ até 200 matérias por câmara |
| Campina Grande | — (só publica notícias sobre PLs no WordPress) | ✗ sem dado estruturado |


In [8]:

MAX_PAGINAS_SAPL    = 20   
MAX_PROJETOS_CIDADE = 1000


def _p_sapl(base_sapl: str) -> list[dict]:
    """Adapter genérico para qualquer instância SAPL (Interlegis).

    Estrutura observada: a página de pesquisa retorna uma <table> com uma <tr>
    por matéria. Cada <tr> tem <a href='/materia/<id>'> e o texto contém:
        TIPO N/AAAA - NomeTipo Ementa: ...  Apresentação: ...  Autor: ...
    """
    out = []
    base_query = f"{base_sapl}/materia/pesquisar-materia?salvar=Pesquisar"
    for page in range(1, MAX_PAGINAS_SAPL + 1):
        url = base_query + (f"&page={page}" if page > 1 else "")
        try:
            html = fetch(url)
        except FetchError as exc:
            log.warning("SAPL %s pg %d falhou: %s", base_sapl, page, exc)
            break
        if not html:
            break
        soup = soup_of(html)
        achados_pagina = 0
        for tr in soup.find_all("tr"):
            a = tr.find("a", href=re.compile(r"/materia/\d+"))
            if not a:
                continue
            texto = tr.get_text(" ", strip=True)
            m_id = re.match(r"^([A-ZÀ-Ú]+)\s+(\d+)\s*/\s*(\d{4})", texto)
            if not m_id:
                continue
            m_em = re.search(r"Ementa:\s*(.*?)\s*(?:Apresenta[çc][ãa]o:|Autor:|$)", texto)
            m_au = re.search(r"Autor:\s*(.*?)(?:\s*Texto Original|\s*Localiza[çc][ãa]o|\s{3,}|$)", texto)
            out.append({
                "tipo":      m_id.group(1),
                "numero":    int(m_id.group(2)),
                "ano":       int(m_id.group(3)),
                "ementa":    (m_em.group(1) if m_em else "").strip()[:400],
                "autor":     (m_au.group(1) if m_au else "").strip()[:120],
                "url_texto": urljoin(base_sapl, a["href"]),
            })
            achados_pagina += 1
            if len(out) >= MAX_PROJETOS_CIDADE:
                return out
        if achados_pagina == 0:
            break  # parou de paginar
    return out


def _p_patos_json(url: str) -> list[dict]:
    """Patos: o JS inline da página de protocolo expõe `const projetosDeLei = [...]`."""
    try:
        html = fetch(url)
    except FetchError as exc:
        log.error("Patos: falha (%s)", exc); return []
    m = re.search(r"const\s+projetosDeLei\s*=\s*(\[.*?\]);", html, re.DOTALL)
    if not m:
        log.error("Patos: array projetosDeLei não encontrado no HTML"); return []
    try:
        data = json.loads(m.group(1))
    except json.JSONDecodeError as exc:
        log.error("Patos: JSON inválido (%s)", exc); return []
    out = []
    for item in data:
        num_ano = item.get("numero", "")
        m_na = re.match(r"(\d{4})/(\d{1,5})", num_ano) or re.match(r"(\d{1,5})/(\d{4})", num_ano)
        if not m_na: continue
        # formato em Patos é ANO/NUMERO
        if int(m_na.group(1)) > 1900:
            ano, numero = int(m_na.group(1)), int(m_na.group(2))
        else:
            numero, ano = int(m_na.group(1)), int(m_na.group(2))
        url_arq = None
        arq = item.get("arquivos")
        if isinstance(arq, str) and arq.startswith("["):
            try:    url_arq = eval(arq)[0]
            except: pass
        elif isinstance(arq, list) and arq:
            url_arq = arq[0]
        out.append({
            "tipo":      "PL",
            "numero":    numero, "ano": ano,
            "ementa":    (item.get("assunto") or "").strip()[:400],
            "autor":     (item.get("remetente") or "").strip()[:120],
            "url_texto": url_arq,
            "situacao":  item.get("situacao"),
        })
    return out


def _p_indisponivel(cidade: CamaraConfig) -> list[dict]:
    log.warning("%s: sem rota estruturada de projetos em HTML estático", cidade.slug)
    return []


def extrair_projetos(cidade: CamaraConfig) -> list[dict]:
    if cidade.sapl_base:
        registros = _p_sapl(cidade.sapl_base)
    elif cidade.url_json_projetos:
        registros = _p_patos_json(cidade.url_json_projetos)
    else:
        registros = _p_indisponivel(cidade)
    for r in registros:
        r["municipio"]  = cidade.municipio
        r["slug"]       = cidade.slug
        r["url_origem"] = cidade.sapl_base or cidade.url_projetos
    log.info("%-15s projetos extraídos: %d", cidade.slug, len(registros))
    return registros


projetos_raw: list[dict] = []
for c in tqdm(CIDADES, desc="Projetos"):
    projetos_raw.extend(extrair_projetos(c))

df_projetos_raw = pd.DataFrame(projetos_raw)
print(f"Total: {len(df_projetos_raw)} projetos")
df_projetos_raw.head(10)


Projetos:   0%|          | 0/5 [00:00<?, ?it/s]

16:41:24 [INFO] pauta.scraper — joao_pessoa     projetos extraídos: 786


Projetos:  20%|██        | 1/5 [01:47<07:10, 107.66s/it]

16:42:04 [WARNING] pauta.scraper — SAPL https://sapl.campinagrande.pb.leg.br pg 11 falhou: HTTP 403 em https://sapl.campinagrande.pb.leg.br/materia/pesquisar-materia?salvar=Pesquisar&page=11 (final: https://sapl.campinagrande.pb.leg.br/materia/pesquisar-materia?salvar=Pesquisar&page=11)


16:42:04 [INFO] pauta.scraper — campina_grande  projetos extraídos: 500


...

16:43:00 [INFO] pauta.scraper — bayeux          projetos extraídos: 601


Projetos: 100%|██████████| 5/5 [03:23<00:00, 30.36s/it]

Projetos: 100%|██████████| 5/5 [03:23<00:00, 40.79s/it]

Total: 2528 projetos


,tipo,numero,ano,ementa,autor,url_texto,municipio,slug,url_origem,situacao
0,VETO,31,2026,VETO PARCIAL AO PROJETO DE LEI ORDINÁRIA Nº 57...,Prefeito Cícero Lucena - Prefeito Relatorias: ...,https://sapl.joaopessoa.pb.leg.br/materia/194535,João Pessoa,joao_pessoa,https://sapl.joaopessoa.pb.leg.br,NaN
1,VETO,32,2026,VETO TOTAL AO PROJETO DE LEI Nº 93/2025 (AUTÓG...,Prefeito Cícero Lucena - Prefeito Relatorias: ...,https://sapl.joaopessoa.pb.leg.br/materia/194950,João Pessoa,joao_pessoa,https://sapl.joaopessoa.pb.leg.br,NaN
2,VETO,33,2026,VETO TOTAL AO PROJETO DE LEI ORDINÁRIA Nº 147/...,Prefeito Cícero Lucena - Prefeito Relatorias: ...,https://sapl.joaopessoa.pb.leg.br/materia/194955,João Pessoa,joao_pessoa,https://sapl.joaopessoa.pb.leg.br,NaN
3,VETO,34,2026,VETO TOTAL AO PROJETO DE LEI Nº 246/2025 (AUTÓ...,Prefeito Cícero Lucena - Prefeito Relatorias: ...,https://sapl.joaopessoa.pb.leg.br/materia/194957,João Pessoa,joao_pessoa,https://sapl.joaopessoa.pb.leg.br,NaN
4,VETO,35,2026,VETO TOTAL AO PROJETO DE LEI Nº 287/2025 (AUTÓ...,Prefeito Cícero Lucena - Prefeito Relatorias: ...,https://sapl.joaopessoa.pb.leg.br/materia/194959,João Pessoa,joao_pessoa,https://sapl.joaopessoa.pb.leg.br,NaN
5,VETO,36,2026,VETO TOTAL AO PROJETO DE LEI Nº 291/2025 (AUTÓ...,Prefeito Cícero Lucena - Prefeito Relatorias: ...,https://sapl.joaopessoa.pb.leg.br/materia/194960,João Pessoa,joao_pessoa,https://sapl.joaopessoa.pb.leg.br,NaN
6,VETO,37,2026,,Prefeito Cícero Lucena - Prefeito Relatorias: ...,https://sapl.joaopessoa.pb.leg.br/materia/194963,João Pessoa,joao_pessoa,https://sapl.joaopessoa.pb.leg.br,NaN
7,VETO,38,2026,VETO PARCIAL AO PROJETO DE LEI NO 489/2025 (AU...,,https://sapl.joaopessoa.pb.leg.br/materia/194964,João Pessoa,joao_pessoa,https://sapl.joaopessoa.pb.leg.br,NaN
8,VETO,39,2026,VETO TOTAL AO PROJETO DE LEI ORDINÁRIA Nº 205/...,Prefeito Cícero Lucena - Prefeito Relatorias: ...,https://sapl.joaopessoa.pb.leg.br/materia/194966,João Pessoa,joao_pessoa,https://sapl.joaopessoa.pb.leg.br,NaN
9,VETO,40,2026,VETO PARCIAL AO AUTÓGRAFO Nº 3968/2025 (MEDIDA...,Prefeito Cícero Lucena - Prefeito Relatorias: ...,https://sapl.joaopessoa.pb.leg.br/materia/195136,João Pessoa,joao_pessoa,https://sapl.joaopessoa.pb.leg.br,NaN



Aqui o foco é metadado leve: título, data (quando aparecer) e URL. O texto completo é coletado sob demanda em etapa futura.

In [9]:
RX_DATA_BR = re.compile(r"\b(\d{1,2})[\/\-](\d{1,2})[\/\-](\d{2,4})\b")
MAX_NOTICIAS_POR_CIDADE = 50


def extrair_noticias(cidade: CamaraConfig) -> list[dict]:
    url = cidade.url_noticias
    if not url:
        return []
    try:
        html = fetch(url)
    except FetchError as exc:
        log.error("%s: falha ao buscar notícias (%s)", cidade.slug, exc)
        return []
    soup = soup_of(html)

    registros: list[dict] = []
    for h in soup.find_all(["h2", "h3", "h4"]):
        a = h.find("a", href=True) or h.find_parent("a", href=True)
        titulo = re.sub(r"\s+", " ", h.get_text(strip=True))
        if not titulo or len(titulo) < 8:
            continue

        contexto = h.parent.get_text(" ", strip=True) if h.parent else titulo
        m = RX_DATA_BR.search(contexto)
        data = f"{m.group(3).zfill(4)}-{m.group(2).zfill(2)}-{m.group(1).zfill(2)}" if m else None

        registros.append({
            "municipio":  cidade.municipio,
            "slug":       cidade.slug,
            "titulo":     titulo,
            "data":       data,
            "url":        urljoin(url, a["href"]) if a else None,
            "url_origem": url,
        })
        if len(registros) >= MAX_NOTICIAS_POR_CIDADE:
            break

    log.info("%-15s notícias extraídas: %d", cidade.slug, len(registros))
    return registros


noticias_raw: list[dict] = []
for c in tqdm(CIDADES, desc="Notícias"):
    noticias_raw.extend(extrair_noticias(c))

df_noticias_raw = pd.DataFrame(noticias_raw)
df_noticias_raw.head(10)


Notícias:   0%|          | 0/5 [00:00<?, ?it/s]

16:43:02 [INFO] pauta.scraper — joao_pessoa     notícias extraídas: 0


Notícias:  20%|██        | 1/5 [00:02<00:09,  2.40s/it]

16:43:05 [INFO] pauta.scraper — campina_grande  notícias extraídas: 15


Notícias:  40%|████      | 2/5 [00:05<00:08,  2.77s/it]

...

Notícias:  80%|████████  | 4/5 [00:19<00:04,  4.95s/it]

16:43:22 [INFO] pauta.scraper — bayeux          notícias extraídas: 1


Notícias: 100%|██████████| 5/5 [00:22<00:00,  4.21s/it]

Notícias: 100%|██████████| 5/5 [00:22<00:00,  4.47s/it]

,municipio,slug,titulo,data,url,url_origem
0,Campina Grande,campina_grande,Lei de autoria de Carol Gomes entra em prática...,None,https://www.camaracg.pb.gov.br/lei-de-autoria-...,https://www.camaracg.pb.gov.br/noticias/
1,Campina Grande,campina_grande,Câmara aprova mais de 200 requerimentos em ses...,None,https://www.camaracg.pb.gov.br/camara-aprova-m...,https://www.camaracg.pb.gov.br/noticias/
2,Campina Grande,campina_grande,"Câmara debate saúde pública, infraestrutura e ...",None,https://www.camaracg.pb.gov.br/camara-debate-s...,https://www.camaracg.pb.gov.br/noticias/
3,Campina Grande,campina_grande,CMCG homenageia Luizinho Calixto com Medalha d...,None,https://www.camaracg.pb.gov.br/cmcg-homenageia...,https://www.camaracg.pb.gov.br/noticias/
4,Campina Grande,campina_grande,Vereadores aprovam requerimentos nas áreas de ...,None,https://www.camaracg.pb.gov.br/vereadores-apro...,https://www.camaracg.pb.gov.br/noticias/
5,Campina Grande,campina_grande,"Defensor da privatização da Cagepa, Alexandre ...",None,https://www.camaracg.pb.gov.br/defensor-da-pri...,https://www.camaracg.pb.gov.br/noticias/
6,Campina Grande,campina_grande,Bancadas se unem em cobrança à Energisa e pede...,None,https://www.camaracg.pb.gov.br/bancadas-se-une...,https://www.camaracg.pb.gov.br/noticias/
7,Campina Grande,campina_grande,CMCG homenageia Laécio Fama e Tâmela Fama em S...,None,https://www.camaracg.pb.gov.br/cmcg-homenageia...,https://www.camaracg.pb.gov.br/noticias/
8,Campina Grande,campina_grande,Câmara debate parceria público-privada da Cagepa,None,https://www.camaracg.pb.gov.br/camara-debate-p...,https://www.camaracg.pb.gov.br/noticias/
9,Campina Grande,campina_grande,Projeto da Vereadora Carol Gomes que objetiva ...,None,https://www.camaracg.pb.gov.br/projeto-da-vere...,https://www.camaracg.pb.gov.br/noticias/



- **Normalização de strings**: trim, colapsar espaços, padronizar caixa.
- **Deduplicação**: chaves de negócio claras (`municipio + nome` para vereadores; `slug + tipo + numero + ano` para projetos; `municipio + titulo` para notícias).
- **Tipos corretos**: `int` em `numero`/`ano`, `datetime` em `data`.
- **Drop de linhas sem dado-chave**.



In [10]:
def normalize_str(s):
    if pd.isna(s):
        return s
    return re.sub(r"\s+", " ", str(s)).strip()


# ----- Vereadores -----
if not df_vereadores_raw.empty:
    df_vereadores = df_vereadores_raw.copy()
    for col in ("nome", "partido", "email", "telefone", "bio", "foto", "url_perfil"):
        if col in df_vereadores:
            df_vereadores[col] = df_vereadores[col].map(normalize_str)
    df_vereadores = (
        df_vereadores.dropna(subset=["nome"])
                     .drop_duplicates(subset=["municipio", "nome"], keep="first")
                     .reset_index(drop=True)
    )
else:
    df_vereadores = df_vereadores_raw

# ----- Projetos -----
if not df_projetos_raw.empty:
    df_projetos = df_projetos_raw.copy()
    df_projetos["ementa"] = df_projetos["ementa"].map(normalize_str)
    df_projetos["numero"] = pd.to_numeric(df_projetos["numero"], errors="coerce").astype("Int64")
    df_projetos["ano"]    = pd.to_numeric(df_projetos["ano"],    errors="coerce").astype("Int64")
    df_projetos = (
        df_projetos.dropna(subset=["tipo", "numero", "ano"])
                   .drop_duplicates(subset=["slug", "tipo", "numero", "ano"], keep="first")
                   .reset_index(drop=True)
    )
else:
    df_projetos = df_projetos_raw

# ----- Notícias -----
if not df_noticias_raw.empty:
    df_noticias = df_noticias_raw.copy()
    df_noticias["titulo"] = df_noticias["titulo"].map(normalize_str)
    df_noticias["data"]   = pd.to_datetime(df_noticias["data"], errors="coerce")
    df_noticias = (
        df_noticias.dropna(subset=["titulo"])
                   .drop_duplicates(subset=["municipio", "titulo"], keep="first")
                   .reset_index(drop=True)
    )
else:
    df_noticias = df_noticias_raw

print(f"Vereadores: {len(df_vereadores_raw)} bruto → {len(df_vereadores)} limpo")
print(f"Projetos:   {len(df_projetos_raw)} bruto → {len(df_projetos)} limpo")
print(f"Notícias:   {len(df_noticias_raw)} bruto → {len(df_noticias)} limpo")

Vereadores: 97 bruto → 97 limpo
Projetos:   2528 bruto → 2486 limpo
Notícias:   57 bruto → 47 limpo


---
## Fase 9 — Agregação: vereador + suas propostas (insumo do recomendador)

Produz o artefato central da Task 1: **uma linha por vereador, com tudo agregado**.
Duas representações das propostas, pra cobrir casos de uso distintos:

| Campo | Tipo | Uso |
|-------|------|-----|
| `corpus_pautas`   | string concatenada (`ementa1 \| ementa2 \| ...`) | embedding único do vereador (Task 2) |
| `propostas_json`  | JSON com lista de `{tipo, numero, ano, ementa, url_texto}` | análise estruturada, recomendação por proposta, debugging |

Etapas:
1. **Match autor → vereador** (fuzzy por tokens significativos).
2. **Agregar propostas** ordenadas (mais recentes primeiro).
3. **Tabela final** salva como CSV + JSONL (1 vereador por linha em JSON).


In [11]:
import unicodedata
from collections import defaultdict

def _norm_nome(s: str) -> str:
    """Normaliza nome pra match fuzzy: minúsculo, sem acento, sem pontuação."""
    if not s: return ""
    s = unicodedata.normalize("NFD", s).encode("ascii", "ignore").decode().lower()
    s = re.sub(r"[^a-z\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def _tokens_significativos(nome: str) -> set[str]:
    """Conjunto de tokens >=4 chars (ignora 'de', 'da', 'do', etc)."""
    stops = {"de","da","do","das","dos","e","silva","santos","sousa","souza","pereira","junior","filho","neto"}
    return {t for t in _norm_nome(nome).split() if len(t) >= 4 and t not in stops}

def _apelido(nome: str) -> str | None:
    """Extrai apelido entre parênteses: 'Severino Farias da França (Farias)' → 'farias'."""
    m = re.search(r"\(([^)]+)\)", nome or "")
    return _norm_nome(m.group(1)) if m else None

def linkar_autor_a_vereador(autor: str, vereadores_da_cidade: list[dict]) -> dict | None:
    """Match em 2 estratégias:
    1) Apelido entre parênteses do vereador bate exatamente com o autor (normalizado)
       → resolve casos tipo autor='Farias' / vereador='Severino Farias da França (Farias)'
    2) Fallback: similaridade fuzzy por tokens significativos (limiar 50%).
    """
    if not autor: return None
    autor_norm = _norm_nome(autor)
    # 1) Match por apelido exato
    for v in vereadores_da_cidade:
        ap = _apelido(v["nome"])
        if ap and ap == autor_norm:
            return v
    # 2) Fuzzy por tokens (incluindo tokens do apelido)
    tokens_autor = _tokens_significativos(autor)
    if not tokens_autor: return None
    melhor, melhor_score = None, 0
    for v in vereadores_da_cidade:
        # combina tokens do nome completo + apelido
        tokens_v = _tokens_significativos(v["nome"])
        ap = _apelido(v["nome"])
        if ap: tokens_v |= _tokens_significativos(ap)
        if not tokens_v: continue
        inter = tokens_autor & tokens_v
        # score relativo ao MENOR dos dois conjuntos — autor curto (só apelido) ainda casa
        score = len(inter) / max(min(len(tokens_v), len(tokens_autor)), 1)
        if score > melhor_score and score >= 0.5:
            melhor, melhor_score = v, score
    return melhor

# Linka cada projeto ao vereador autor (quando o autor for um vereador da câmara)
projetos_linkados = []
for p_ in projetos_raw:
    autor = p_.get("autor", "")
    vereadores_cidade = [v for v in vereadores_raw if v["slug"] == p_["slug"]]
    v_link = linkar_autor_a_vereador(autor, vereadores_cidade)
    p_["vereador_nome"] = v_link["nome"] if v_link else None
    projetos_linkados.append(p_)

df_projetos_linkados = pd.DataFrame(projetos_linkados)
n_linkados = df_projetos_linkados["vereador_nome"].notna().sum()
print(f"Projetos com autor identificado como vereador: {n_linkados}/{len(df_projetos_linkados)}")

# Tabela final agregada por vereador
agregado = []
for v in vereadores_raw:
    propostas = [
        p_ for p_ in projetos_linkados
        if p_["slug"] == v["slug"] and p_.get("vereador_nome") == v["nome"]
    ]
    propostas_estruturadas = [
        {
            "tipo":      p_["tipo"],
            "numero":    p_["numero"],
            "ano":       p_["ano"],
            "ementa":    p_.get("ementa", ""),
            "url_texto": p_.get("url_texto"),
        }
        for p_ in propostas
    ]
    # ordena por (ano desc, número desc) — mais recentes primeiro
    propostas_estruturadas.sort(key=lambda x: (-(x["ano"] or 0), -(x["numero"] or 0)))
    ementas = [p_["ementa"] for p_ in propostas_estruturadas if p_.get("ementa")]
    corpus = " | ".join(ementas)
    # corpus = bio + todas as ementas (insumo do embedding na Task 2)
    if v.get("bio"):
        corpus = (v["bio"] + " || " + corpus).strip(" |")
    agregado.append({
        "municipio":   v["municipio"],
        "slug":        v["slug"],
        "nome":        v["nome"],
        "partido":     v.get("partido"),
        "telefone":    v.get("telefone"),
        "email":       v.get("email"),
        "foto":        v.get("foto"),
        "url_perfil":  v.get("url_perfil"),
        "tem_bio":     bool(v.get("bio")),
        "n_propostas": len(propostas),
        "tipos_propostas": ", ".join(sorted({p_["tipo"] for p_ in propostas})),
        "propostas_json":  json.dumps(propostas_estruturadas, ensure_ascii=False),
        "corpus_pautas":   corpus[:5000],   # versão string p/ embedding direto
    })

df_vereador_pauta = pd.DataFrame(agregado).sort_values(["municipio", "n_propostas"], ascending=[True, False]).reset_index(drop=True)
print(f"\nVereadores com 1+ proposta: {(df_vereador_pauta['n_propostas']>0).sum()}/{len(df_vereador_pauta)}")
print(f"Corpus médio: {df_vereador_pauta['corpus_pautas'].str.len().mean():.0f} chars")
df_vereador_pauta[["municipio","nome","partido","n_propostas","tipos_propostas"]].head(15)


Projetos com autor identificado como vereador: 2100/2528

Vereadores com 1+ proposta: 94/97
Corpus médio: 2756 chars


,municipio,nome,partido,n_propostas,tipos_propostas
0,Bayeux,Cabo Rubem,PSB,112,"EIMP, PDC, PDL, PLC, PLO, PR, REQ, VET"
1,Bayeux,Jays de Nita,PSB,74,"ATOPR, CONV, DESPR, EDIT, EIMP, PLO, REQ"
2,Bayeux,Josauro Pereira,MDB,56,"EIMP, IND, PDL, PLO, REQ"
3,Bayeux,Wagner do Grau,PSD,41,"EIMP, EMD, INSLE, PLO, REQ, VET"
4,Bayeux,França,PT,37,"EIMP, IND, PLO, REQ"
5,Bayeux,Pastora Anunciada,PSB,32,"EIMP, PDL, PLO, REQ, REQAD"
6,Bayeux,Adriano do Táxi,PSB,31,"EIMP, IND, PDL, PLO, PR, REQ"
7,Bayeux,Iara Caetano,REPUBLICANOS,31,"EIMP, IND, PLO, REQ, RQAPR"
8,Bayeux,Berguinho Impacto Som,PV,26,"EIMP, PLO, REQ"
9,Bayeux,Nildo da Casa Branca,MOBILIZA,22,"EIMP, ETA, PLO, PR, REQ"




| Formato | Arquivo | Uso |
|---------|---------|-----|
| **JSONL** | `vereador_pauta.jsonl` | Task 2 (NLP+LLM) — 1 vereador/linha, `propostas` aninhadas |
| **CSV agregado** | `vereador_pauta.csv` | 1 linha por vereador (`propostas_json` serializado) |
| **CSV explodido** | `vereador_propostas_long.csv` | Planilha/Excel — 1 linha por (vereador, proposta) |


In [12]:
# Persistência: apenas os 3 artefatos que alimentam as próximas tasks.
# (vereadores/projetos/notícias/descoberta crus e o SQLite foram removidos —
#  toda a informação necessária já está consolidada na tabela vereador_pauta.)

# --- 1) JSONL: 1 vereador por linha, com propostas aninhadas (insumo da Task 2) ---
jsonl_path = OUTPUT_DIR / "vereador_pauta.jsonl"
with open(jsonl_path, "w", encoding="utf-8") as f:
    for _, row in df_vereador_pauta.iterrows():
        registro = row.to_dict()
        if registro.get("propostas_json"):
            registro["propostas"] = json.loads(registro["propostas_json"])
        registro.pop("propostas_json", None)
        f.write(json.dumps(registro, ensure_ascii=False, default=str) + "\n")

# --- 2) CSV agregado: 1 linha por vereador (propostas serializadas em JSON) ---
csv_agregado = OUTPUT_DIR / "vereador_pauta.csv"
df_vereador_pauta.to_csv(csv_agregado, index=False)

# --- 3) CSV explodido: 1 linha por (vereador, proposta) — formato planilha ---
linhas_explodidas = []
for _, row in df_vereador_pauta.iterrows():
    propostas = json.loads(row["propostas_json"]) if row.get("propostas_json") else []
    base = {
        "municipio": row["municipio"],
        "vereador":  row["nome"],
        "partido":   row["partido"],
        "email":     row.get("email"),
        "telefone":  row.get("telefone"),
    }
    if propostas:
        for prop in propostas:
            linhas_explodidas.append({**base,
                "proposta_tipo":   prop["tipo"],
                "proposta_numero": prop["numero"],
                "proposta_ano":    prop["ano"],
                "proposta_ementa": prop["ementa"],
                "proposta_url":    prop.get("url_texto"),
            })
    else:
        linhas_explodidas.append({**base,
            "proposta_tipo": None, "proposta_numero": None,
            "proposta_ano": None, "proposta_ementa": None, "proposta_url": None,
        })
csv_explodido = OUTPUT_DIR / "vereador_propostas_long.csv"
pd.DataFrame(linhas_explodidas).to_csv(csv_explodido, index=False)

for caminho in (jsonl_path, csv_agregado, csv_explodido):
    print(f"{caminho.name:32} → {caminho.stat().st_size/1024:.1f} KB")


vereador_pauta.jsonl             → 894.7 KB
vereador_pauta.csv               → 909.9 KB
vereador_propostas_long.csv      → 584.9 KB
